# EP01 圖轉影片技術驗證 — Wan2.2-TI2V-5B on Colab

用途：用已經核可的厲若楓Reference圖，測試免費Colab GPU能不能跑Wan2.2把靜態圖轉成動態影片。
這是純技術驗證，不是正式Shot生產。

官方建議至少24GB VRAM，免費Colab的T4只有16GB，本測試會用最省顯存的設定（CPU offload、
較短秒數），如果還是OOM（顯存不足）會如實記錄，不強行硬跑。

**使用方式**：
1. 選單「執行階段」→「變更執行階段類型」→ 硬體加速器選 **T4 GPU**（如果帳號有L4或更高階可用更好），儲存
2. 執行階段→中斷連線並刪除執行階段（確保乾淨環境）
3. 從上到下依序執行，Step 3會跳出上傳視窗，選擇厲若楓的參考圖上傳
4. 跑完會產生一段短影片並自動下載

In [ ]:
# Step 1：確認拿到的GPU與VRAM
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Step 2：安裝依賴。Wan2.2官方要求diffusers從GitHub source裝，不能用PyPI版本
!pip install -q git+https://github.com/huggingface/diffusers.git
!pip install -q transformers accelerate safetensors imageio imageio-ffmpeg ftfy

In [ ]:
# Step 3：上傳已核可的厲若楓 Reference 圖
from google.colab import files
uploaded = files.upload()  # 選擇 LI_RUOFENG_REF_01.png
input_image_path = list(uploaded.keys())[0]
print('已上傳：', input_image_path)

In [ ]:
# Step 4：載入Wan2.2-TI2V-5B（官方最小的圖生影片模型，仍建議24GB VRAM，這裡盡量壓省顯存）
import torch
from diffusers import WanPipeline
from diffusers.utils import load_image, export_to_video

pipe = WanPipeline.from_pretrained(
    'Wan-AI/Wan2.2-TI2V-5B-Diffusers', torch_dtype=torch.bfloat16
)
pipe.enable_model_cpu_offload()  # T4只有16GB，官方建議24GB，靠offload硬擠
print('Wan2.2-TI2V-5B 載入完成')

In [ ]:
# Step 5：圖生影片 —— 依MASTER_VISUAL_STYLE_LOCK第十一節的表演規則：
# 自然體態、細微表情、寫實衣料飄動、可控鏡頭，不要突然變臉/瞬移/肢體變形
image = load_image(input_image_path)

prompt = (
    "the character stands calmly and naturally, subtle idle breathing motion, gentle breeze "
    "moving his hair and clothing slightly, calm observant eyes slowly looking to the side, "
    "no sudden movement, no camera shake, static camera, natural cloth physics, cinematic "
    "lighting, preserve exact character identity and costume"
)
negative_prompt = (
    "sudden transformation, teleport, deformed limbs, extra fingers, face swap, costume "
    "change, hairstyle change, watermark, text, low quality, blurry"
)

# 顯存有限，先用較短秒數＋較低幀數測試（成功再考慮拉長）
output = pipe(
    image=image,
    prompt=prompt,
    negative_prompt=negative_prompt,
    height=704,
    width=1280,
    num_frames=49,   # 約2秒 @24fps，先求跑得動
    guidance_scale=5.0,
).frames[0]

export_to_video(output, 'li_ruofeng_test.mp4', fps=24)
print('影片生成完成：li_ruofeng_test.mp4')

In [ ]:
# Step 6：下載結果
from google.colab import files
files.download('li_ruofeng_test.mp4')